# 01 - Dataset Inspection
**Phase 1 - ML component, Smart Agricultural Water Management System**

Goal: understand what data exists, how tables relate, and whether a reliable target variable exists for
*rice irrigation water requirement*. This notebook only **reads** the database; nothing is modified.

Run all cells top to bottom (Kernel > Restart & Run All).

In [1]:
import sqlite3
from pathlib import Path
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.max_rows", 100)

# Look for the real database in every parent folder of this notebook (and in AgriWater/database).
here = Path.cwd().resolve()
found = []
for p in [here, *here.parents]:
    for cand in [p / "AgriWater" / "database" / "agriwater_v2.sqlite", p / "database" / "agriwater_v2.sqlite"]:
        if cand.exists():
            found.append((cand, cand.stat().st_size))
print("Notebook folder:", here)
print("Database files found:")
for c, sz in found:
    print("  ", c, "->", round(sz / 1e6, 2), "MB")
good = [c for c, sz in found if sz > 1_000_000]
assert good, (
    "No real database found (a file under 1 MB is only a Git-LFS placeholder). "
    "Download agriwater_v2.sqlite (~95 MB) from GitHub and put it in <project>/AgriWater/database/. "
    "Check the list printed above.")
DB = good[0]
ROOT = DB.parent.parent.parent if DB.parent.parent.name == "AgriWater" else DB.parent.parent
REPORTS = ROOT / "machine_learning" / "reports"
con = sqlite3.connect(DB)
print("Project root:", ROOT)
print("Database size (MB):", round(DB.stat().st_size / 1e6, 1))

Notebook folder: C:\Users\menna\Downloads\Smart-Agricultural-Water-Management-main33\Smart-Agricultural-Water-Management-main\machine_learning\notebooks
Database files found:
   C:\Users\menna\Downloads\Smart-Agricultural-Water-Management-main33\Smart-Agricultural-Water-Management-main\AgriWater\database\agriwater_v2.sqlite -> 99.4 MB
Project root: C:\Users\menna\Downloads\Smart-Agricultural-Water-Management-main33\Smart-Agricultural-Water-Management-main
Database size (MB): 99.4


## 1. All tables: rows and columns

In [2]:
tables = pd.read_sql("select name from sqlite_master where type='table' order by name", con)["name"].tolist()
overview = []
for t in tables:
    n = con.execute(f'select count(*) from "{t}"').fetchone()[0]
    ncols = len(pd.read_sql(f'PRAGMA table_info("{t}")', con))
    overview.append({"table": t, "rows": n, "columns": ncols})
overview = pd.DataFrame(overview).sort_values("rows", ascending=False).reset_index(drop=True)
print(len(tables), "tables")
overview

44 tables


,table,rows,columns
0,weather_daily,1089840,6
1,rice_daily_water_demand,46550,6
2,water_scenario_by_variety_v3,9044,22
3,sentinel2_scene,3775,7
4,water_scenario_by_variety_clay,2660,18
5,water_scenario_by_variety,2660,15
6,literature_trial_observation,688,16
7,literature_trial_result,470,13
8,water_scenario,266,16
9,point_season_summary,266,19


### Empty tables (no data yet)

In [3]:
overview[overview.rows == 0]

,table,rows,columns
34,canal_schedule,0,8
35,crop_history,0,6
36,field_season,0,16
37,field_operation,0,7
38,field,0,8
39,irrigation_event,0,6
40,pump_type,0,7
41,groundwater_level,0,6
42,vegetation_index,0,5
43,soil_salinity,0,7


## 2. Relationships between tables (foreign keys)

In [4]:
fk_rows = []
for t in tables:
    fks = pd.read_sql(f'PRAGMA foreign_key_list("{t}")', con)
    for _, r in fks.iterrows():
        fk_rows.append({"table": t, "column": r["from"], "references": f'{r["table"]}.{r["to"]}'})
fk = pd.DataFrame(fk_rows)
print(len(fk), "foreign keys")
fk.sort_values(["references", "table"]).reset_index(drop=True)

48 foreign keys


,table,column,references
0,crop_history,crop_code,crop_reference.crop_code
1,canal_schedule,source_id,data_source.source_id
2,groundwater_level,source_id,data_source.source_id
3,sentinel2_scene,source_id,data_source.source_id
4,soil_salinity,source_id,data_source.source_id
5,weather_daily,source_id,data_source.source_id
6,canal_schedule,district_id,district.district_id
7,district,parent_id,district.district_id
8,field,district_id,district.district_id
9,grid_point,district_id,district.district_id


Reading the keys: weather, soil and the season summary all join on `point_id` (38 grid points) and,
for the season tables, on `season_year`. The published-trial tables (`literature_*`) link to `literature_study`
only and have **no `point_id`**, so they cannot be joined row-by-row to weather or soil.

## 3. Weather data (NASA POWER, gridded estimates)

In [5]:
w = pd.read_sql("""
    select parameter, unit, count(*) n, count(value) non_null,
           min(value) vmin, max(value) vmax, min(obs_date) first_date, max(obs_date) last_date
    from weather_daily group by parameter, unit order by parameter""", con)
print("weather_daily rows:", con.execute("select count(*) from weather_daily").fetchone()[0])
print("grid points:", con.execute("select count(distinct point_id) from weather_daily").fetchone()[0])
w

weather_daily rows: 1089840
grid points: 38


,parameter,unit,n,non_null,vmin,vmax,first_date,last_date
0,ALLSKY_SFC_SW_DWN,MJ/m^2/day,136230,136002,2.07,30.35,2017-01-01,2026-10-25
1,PRECTOTCORR,mm/day,136230,136078,0.00,485.52,2017-01-01,2026-10-25
2,PS,kPa,136230,136078,99.78,103.02,2017-01-01,2026-10-25
3,RH2M,%,136230,136078,20.47,90.73,2017-01-01,2026-10-25
4,T2M,C,136230,136078,8.21,34.80,2017-01-01,2026-10-25
5,T2M_MAX,C,136230,136078,11.23,46.95,2017-01-01,2026-10-25
6,T2M_MIN,C,136230,136078,1.94,28.03,2017-01-01,2026-10-25
7,WS2M,m/s,136230,136078,0.75,10.60,2017-01-01,2026-10-25


In [6]:
# Which parameter/unit combinations hold extreme values? (check units before any cleaning decision)
pd.read_sql("""select parameter, unit, round(max(value),2) max_value, count(*) n_rows_above_100
               from weather_daily where value > 100 group by parameter, unit""", con)

,parameter,unit,max_value,n_rows_above_100
0,PRECTOTCORR,mm/day,485.52,74
1,PS,kPa,103.02,136040


## 4. Soil data (HWSD2, coarse global map - not field samples)

In [7]:
soil = pd.read_sql("select * from soil_point", con)
print(soil.shape)
print(soil.soil_data_status.value_counts(dropna=False).to_dict())
soil.describe().T[["count", "min", "max"]]

(38, 15)
{'COMPLETE': 36, 'INCOMPLETE': 1, 'INCOMPLETE_NODATA_SENTINEL': 1}


,count,min,max
smu_id,38.0,7001.00,16511.000
top_depth_cm,38.0,0.00,0.000
bottom_depth_cm,38.0,20.00,20.000
sand_pct,36.0,21.00,50.000
silt_pct,36.0,24.00,40.000
clay_pct,36.0,20.00,55.000
bulk,36.0,1.33,1.640
organic_carbon,36.0,0.62,0.901
ph_water,36.0,7.30,8.500


## 5. The water 'scenario' tables - what are they?

In [8]:
names = ["water_scenario", "water_scenario_by_variety", "water_scenario_by_variety_clay", "water_scenario_by_variety_v3"]
sc = pd.DataFrame([{"table": n, "rows": con.execute(f"select count(*) from {n}").fetchone()[0]} for n in names])
print(sc)
n3 = sc[sc.table.str.startswith("water_scenario_by_variety")].rows.sum()
print("by_variety + clay + v3 =", n3, "(this is the '14,364 water scenarios' in the task description)")

                            table  rows
0                  water_scenario   266
1       water_scenario_by_variety  2660
2  water_scenario_by_variety_clay  2660
3    water_scenario_by_variety_v3  9044
by_variety + clay + v3 = 14364 (this is the '14,364 water scenarios' in the task description)


In [9]:
# What do the scenario rows say about themselves?
print(pd.read_sql("select actual_irrigation_status, method_status, count(*) n from water_scenario group by 1,2", con).to_string(index=False))
print()
print(pd.read_sql("select distinct assumption_note from water_scenario limit 3", con).to_string(index=False))

actual_irrigation_status                         method_status   n
            NOT_MEASURED SCENARIO_ESTIMATE_NOT_FIELD_VALIDATED 266

                                                                                                    assumption_note
All rainfall treated as effective; Kc, seepage, land preparation and establishment values require local validation.


In [10]:
# The three by_variety tables describe the same points/years/varieties with different assumptions.
keys = "season_year, point_id, variety_id"
for n in names[1:]:
    d = pd.read_sql(f"select count(*) rows, count(distinct season_year||'-'||point_id||'-'||variety_id) distinct_point_year_variety from {n}", con)
    print(n, d.to_dict("records")[0])

water_scenario_by_variety {'rows': 2660, 'distinct_point_year_variety': 1330}
water_scenario_by_variety_clay {'rows': 2660, 'distinct_point_year_variety': 1330}
water_scenario_by_variety_v3 {'rows': 9044, 'distinct_point_year_variety': 4522}


## 6. Is there a real (measured) target variable?

In [11]:
checks = {
    "field_season rows (planned real irrigation labels)": "select count(*) from field_season",
    "irrigation_event rows": "select count(*) from irrigation_event",
    "v_observed_labels rows (real measurements joined to points)": "select count(*) from v_observed_labels",
    "field rows": "select count(*) from field",
}
for k, q in checks.items():
    print(f"{k}: {con.execute(q).fetchone()[0]}")
print()
print(pd.read_sql("select water_values_status, training_readiness, count(*) n from point_season_summary group by 1,2", con).to_string(index=False))

field_season rows (planned real irrigation labels): 0
irrigation_event rows: 0
v_observed_labels rows (real measurements joined to points): 0
field rows: 0

                   water_values_status                    training_readiness   n
MODEL_ESTIMATE_NOT_OBSERVED_IRRIGATION FEATURES_READY_TARGET_LABELS_REQUIRED 266


## 7. Published trial data (closest thing to measured irrigation)

In [12]:
obs = pd.read_sql("""
    select study_id, metric_name, unit, count(*) n, min(metric_value) vmin, max(metric_value) vmax
    from literature_trial_observation
    where metric_name like '%irrigation%' or unit like '%m3%' group by 1,2,3""", con)
obs

,study_id,metric_name,unit,n,vmin,vmax
0,ABOU_EL_HASSAN_KARADA_WATER_DEFICIT_2014_2015,total_water_applied,m3/ha,8,13469.000,14842.000
1,ABOU_EL_HASSAN_KARADA_WATER_DEFICIT_2014_2015,water_use_efficiency,kg/m3,8,0.760,0.819
2,DARWESH_SAKHA_IRRIGATION_DEPTH_INTERVAL_2013_2014,bulk_density,kg/m3_as_printed_scale_ambiguous,4,1.160,1.210
3,DARWESH_SAKHA_IRRIGATION_DEPTH_INTERVAL_2013_2014,seasonal_irrigation_applied,m3/feddan,36,2690.000,6780.000
4,DARWESH_SAKHA_IRRIGATION_DEPTH_INTERVAL_2013_2014,water_productivity,kg/m3,36,0.560,0.960
5,EID_NORTH_DELTA_RICE_PLANTING_DEPTH_2023_2024,irrigation_water_applied,m3/ha,18,8473.000,14689.000
6,EID_NORTH_DELTA_RICE_PLANTING_DEPTH_2023_2024,irrigation_water_productivity,kg/m3,18,0.880,1.180
7,ELHENAWY_KFS_RICE_PREVIOUS_CROP_2007,bulk_density,Mg/m3,14,1.107,1.312
8,GHAZY_SAKHA106_IRRIGATION_COMPOST_SILICATE_2022_2023,total_irrigation_water,m3/ha,6,9435.000,13005.000
9,GHAZY_SAKHA106_IRRIGATION_COMPOST_SILICATE_2022_2023,water_use_efficiency,kg/m3,6,0.761,0.898


In [13]:
pd.read_sql("""
    select s.study_id, s.location_name, s.publication_year, count(o.observation_id) n_observations
    from literature_study s left join literature_trial_observation o using(study_id)
    group by 1,2,3 order by n_observations desc""", con)

,study_id,location_name,publication_year,n_observations
0,DARWESH_SAKHA_IRRIGATION_DEPTH_INTERVAL_2013_2014,"Sakha Agricultural Research Station Farm, Kafr El-Sheikh",2016,288
1,MOURSI_SAKHA_RICE_WATER_COMPONENTS_2012_2013,"Sakha Agricultural Research Station, Kafr El-Sheikh",2015,144
2,ELHENAWY_KFS_RICE_PREVIOUS_CROP_2007,"Faculty of Agriculture farm, Kafrelsheikh University; ex...",2013,104
3,EID_NORTH_DELTA_RICE_PLANTING_DEPTH_2023_2024,"Northern Nile Delta, Egypt; exact station/location not c...",2025,54
4,ABOU_EL_HASSAN_KARADA_WATER_DEFICIT_2014_2015,"El-Karada Research Station, Kafr El-Sheikh",2017,36
5,GHAZY_SAKHA106_IRRIGATION_COMPOST_SILICATE_2022_2023,"Experimental Farm, Sakha Agriculture Research Station, K...",2024,22
6,GEWAILY_SAKHA_RICE_SOIL_BASELINE_2016_2017,"Rice Research and Training Center experimental farm, Sak...",2018,20
7,IBRAHIM_TAG_ELEZZ_RICE_PREVIOUS_CROP_2000_2001,"Tag El-Ezz Agricultural Research Station, Dakahlia Gover...",2002,20
8,YAGETA_SAKHA_PREVIOUS_RICE_SALINITY_2010,15 study sites within 5 km of the Rice Research and Trai...,2013,0
9,eid_hammoul_rice_water_soil_2020_2021,"Hammoul District, North Nile Delta, Egypt",2022,0


These are **treatment means from research-station experiments** in a few studies - not farmer-field
measurements, and not linked to our 38 grid points. They cannot be used as a normal supervised-learning label set.

## 8. Duplicates

In [14]:
dup = pd.read_sql("select item, details, count(*) n from database_metadata group by item, details having n > 1", con)
print("exact duplicate groups in database_metadata:", len(dup))
dup

exact duplicate groups in database_metadata: 4


,item,details,n
0,darwesh_sakha_trial_2013_2014,Imported 288 transcribed table values: irrigation/yield/...,4
1,kfs_previous_crop_soil_trial,Imported 104 published treatment/crop/depth means from E...,4
2,sakha_previous_rice_history_evidence,One published 15-site study-level evidence record from n...,3
3,sakha_soil_baseline_2016_2017,Imported 20 published soil property/nutrient values from...,3


## 9. Summary of ML suitability (from the inventory file)

In [15]:
inv = pd.read_excel(REPORTS / "dataset_inventory.xlsx")
print(inv["ML Suitability"].value_counts().to_string())
inv[["Dataset Name", "Number of Records", "ML Suitability"]].sort_values("ML Suitability")

ML Suitability
Unsuitable    22
Uncertain     17
Suitable       5


,Dataset Name,Number of Records,ML Suitability
2,grid_point,38,Suitable
3,season,7,Suitable
9,weather_daily,1089840,Suitable
11,soil_point,38,Suitable
13,point_season_summary,266,Suitable
0,data_source,21,Uncertain
40,literature_trial_observation,688,Uncertain
37,rice_variety_field_guide,14,Uncertain
36,agronomy_recommendation,23,Uncertain
33,literature_trial_result,470,Uncertain


## 10. Conclusions of the inspection
1. **Inputs exist**: 1,089,840 daily weather rows (38 points, 2017-2026) and soil properties for the 38 points.
2. **No measured irrigation target exists**: `field_season`, `irrigation_event` and `v_observed_labels` are empty.
   All 14,364 scenario rows are **calculated** from ET and assumptions, not measured irrigation.
3. A model trained on the scenarios would only learn the calculation formula (the inputs are the formula's inputs) -
   it would not predict real farm irrigation.
4. The only measured-style irrigation numbers are treatment means in a few published station trials.
5. Next: write the feasibility assessment (`feasibility_assessment.md`) and agree the way forward with the team.